In [1]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import arrays_zip, explode, col

In [2]:
spark = SparkSession.builder \
        .appName("Explode") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/03 11:54:42 WARN Utils: Your hostname, ast-ubuntu, resolves to a loopback address: 127.0.1.1; using 192.168.1.10 instead (on interface wlp2s0)
26/04/03 11:54:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/03 11:54:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
data = [("A", ["x", "y"], [1, 2])]
df = spark.createDataFrame(data, ["id", "letters", "numbers"])
df.show()

df.select(
    "id",
    explode("letters").alias("letter"),
    explode("numbers").alias("number")
).show()

+---+-------+-------+
| id|letters|numbers|
+---+-------+-------+
|  A| [x, y]| [1, 2]|
+---+-------+-------+

+---+------+------+
| id|letter|number|
+---+------+------+
|  A|     x|     1|
|  A|     x|     2|
|  A|     y|     1|
|  A|     y|     2|
+---+------+------+



In [4]:

df.select(
    "id",
    explode(arrays_zip("letters", "numbers")).alias("zipped")
).show() 


+---+------+
| id|zipped|
+---+------+
|  A|{x, 1}|
|  A|{y, 2}|
+---+------+



In [24]:
df.select(
    "id",
    explode(arrays_zip("letters", "numbers")).alias("zipped")
).select(
    "id",
    col("zipped.letters").alias("letter"),
    col("zipped.numbers").alias("number")
).show(truncate=False)


+---+------+------+
|id |letter|number|
+---+------+------+
|A  |x     |1     |
|A  |y     |2     |
+---+------+------+



In [25]:
df.selectExpr(
    "id",
    "explode(arrays_zip(letters, numbers)) as zipped"
).selectExpr(
    "id",
    "zipped.letters as letter",
    "zipped.numbers as number"
).show()


+---+------+------+
| id|letter|number|
+---+------+------+
|  A|     x|     1|
|  A|     y|     2|
+---+------+------+



In [27]:
df.selectExpr(
    "id",
    "explode(arrays_zip(letters, numbers)) as t"
).select(
    "id",
    col("t.letters").alias("letter"),
    col("t.numbers").alias("number")
).show()


+---+------+------+
| id|letter|number|
+---+------+------+
|  A|     x|     1|
|  A|     y|     2|
+---+------+------+



In [20]:
df.selectExpr(
    "id", 
    "explode(arrays_zip(letters, numbers)) as t") \
  .select(
      "id", 
      "t.*") \
  .show()

+---+-------+-------+
| id|letters|numbers|
+---+-------+-------+
|  A|      x|      1|
|  A|      y|      2|
+---+-------+-------+



In [38]:
df.select(
    "id",
    explode(
        arrays_zip(
            col("letters").alias("letter"),
            col("numbers").alias("number")
        )
    ).alias("pair")
).select(
    "id",
    col("pair.letter"),
    col("pair.number")
).show(truncate=False)

+---+------+------+
|id |letter|number|
+---+------+------+
|A  |x     |1     |
|A  |y     |2     |
+---+------+------+



In [42]:
df.selectExpr(
    "id",
    "inline(arrays_zip(letters, numbers))"
).show()

+---+-------+-------+
| id|letters|numbers|
+---+-------+-------+
|  A|      x|      1|
|  A|      y|      2|
+---+-------+-------+



In [4]:
data = [("Akshay", ["English", "Maths"], [60,70], 'Pass')]
df = spark.createDataFrame(data, ["name", "subj", "score", "result"])
df.show()

+------+----------------+--------+------+
|  name|            subj|   score|result|
+------+----------------+--------+------+
|Akshay|[English, Maths]|[60, 70]|  Pass|
+------+----------------+--------+------+



In [6]:
result = df.select(col("name"),
                   explode(arrays_zip(col("subj"),col("score"))).alias("Zipped"),
                   col("result")
                   ) \
            .select(col("name"),
                    col("zipped.subj").alias("Subject"),
                    col("zipped.score").alias("Score"),
                    col("result")
            )

result.show()

+------+-------+-----+------+
|  name|Subject|Score|result|
+------+-------+-----+------+
|Akshay|English|   60|  Pass|
|Akshay|  Maths|   70|  Pass|
+------+-------+-----+------+



In [5]:
# Input data
data = {
    "order_id": 101,
    "items": [
        {"product_id": 1, "price": 100},
        {"product_id": 2, "price": 200}
    ]
}

# Create DataFrame from input data
df = spark.createDataFrame([data])


In [6]:
# Explode the 'items' array to get one row per item
exploded_df = df.select(
    col("order_id"),
    explode(col("items")).alias("item")
)

# Select the required columns
result_df = exploded_df.select(
    col("order_id"),
    col("item.product_id"),
    col("item.price")
)

In [7]:
# Show the result
result_df.show()

+--------+----------+-----+
|order_id|product_id|price|
+--------+----------+-----+
|     101|         1|  100|
|     101|         2|  200|
+--------+----------+-----+



In [8]:
spark.stop()